In [ ]:
# PHẦN 1: CÀI ĐẶT & FIX LỖI "UNPICKLING" (KÈM LORA)

import os
import shutil
import json
import torch
import gc
import time
import nltk
import functools
import numpy as np
from google.colab import drive

os.system("pip install -q peft")
from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from datasets import Dataset

try:
    reconstruct_func = np._core.multiarray._reconstruct if hasattr(np, '_core') else np.core.multiarray._reconstruct
    torch.serialization.add_safe_globals([
        reconstruct_func,
        np.ndarray,
        np.dtype,
    ])
    print("✅ Đã whitelist Numpy cho PyTorch.")
except AttributeError:
    pass

if not hasattr(torch, 'original_load_func'):
    torch.original_load_func = torch.load

def safe_load_override(*args, **kwargs):
    if 'weights_only' in kwargs:
        del kwargs['weights_only']
    return torch.original_load_func(*args, weights_only=False, **kwargs)

torch.load = safe_load_override
print(f"✅ Đã ép buộc torch.load(weights_only=False) thành công.")

os.environ['KAGGLE_USERNAME'] = "phankhaclap"
os.environ['KAGGLE_KEY'] = "0ba946628cb1f5acb76ecd357f590e95"

drive.mount('/content/drive')

FINAL_SAVE_PATH = "/content/drive/My Drive/T5_Small_Spider_CRP_LoRA"
CHECKPOINT_DIR = "/content/drive/My Drive/T5_Small_Spider_CRP_LoRA/checkpoints"

print(">>> [1/5] Tải dữ liệu...")
if os.path.exists('spider_data'): shutil.rmtree('spider_data')
os.system("pip install -q kaggle")
os.system("kaggle datasets download -d jeromeblanchet/yale-universitys-spider-10-nlp-dataset")
os.system("unzip -q yale-universitys-spider-10-nlp-dataset.zip -d temp_spider")

if os.path.exists("temp_spider/spider"):
    shutil.move("temp_spider/spider", "spider_data")
    shutil.rmtree('temp_spider')
    os.system("wget -q https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py")
    os.system("wget -q https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py")
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

# PHẦN 2: TIỀN XỬ LÝ (MỞ RỘNG MAX_LENGTH)

MODEL_NAME = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

with open('spider_data/tables.json', 'r') as f:
    table_data = json.load(f)
schema_map = {db['db_id']: db for db in table_data}

def get_crp_schema(db_id):
    if db_id not in schema_map: return ""
    db = schema_map[db_id]
    ddl_statements = []
    for i, table_name in enumerate(db['table_names_original']):
        col_defs = [f"{c[1]} {db['column_types'][j]}" for j, c in enumerate(db['column_names_original']) if c[0] == i]
        pk_idx = db['primary_keys']
        pks = [db['column_names_original'][j][1] for j in pk_idx if db['column_names_original'][j][0] == i]
        if pks: col_defs.append(f"primary key ({', '.join(pks)})")
        for fk in db['foreign_keys']:
            if db['column_names_original'][fk[0]][0] == i:
                from_col = db['column_names_original'][fk[1]][1]
                to_table = db['table_names_original'][db['column_names_original'][fk[1]][0]]
                to_col = db['column_names_original'][fk[1]][1]
                col_defs.append(f"foreign key ({from_col}) references {to_table}({to_col})")
        ddl_statements.append(f"CREATE TABLE {table_name} ({', '.join(col_defs)});")
    return " ".join(ddl_statements)

def load_spider_unified(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return Dataset.from_dict({
        "question": [item["question"] for item in data],
        "query": [item["query"] for item in data],
        "db_id": [item["db_id"] for item in data]
    })

def preprocess_fn(examples):
    inputs = [f"translate to SQL: {q} | Schema: {get_crp_schema(d)}" for q, d in zip(examples['question'], examples['db_id'])]
    model_inputs = tokenizer(inputs, max_length=1024, padding="max_length", truncation=True)
    labels = tokenizer(examples['query'], max_length=128, padding="max_length", truncation=True)
    labels["input_ids"] = [[(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print(">>> [2/5] Xử lý dữ liệu...")
train_ds = load_spider_unified("spider_data/train_spider.json").map(preprocess_fn, batched=True, desc="Train Prep")
val_ds = load_spider_unified("spider_data/dev.json").map(preprocess_fn, batched=True, desc="Val Prep")

# PHẦN 3: HUẤN LUYỆN VỚI LORA (CẤU HÌNH HẠNG NẶNG)
print(f"\n>>> [3/5] Bắt đầu huấn luyện...")

last_checkpoint = None
if os.path.isdir(CHECKPOINT_DIR):
    checkpoints = [os.path.join(CHECKPOINT_DIR, d) for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint-")]
    if len(checkpoints) > 0:
        checkpoints.sort(key=lambda x: int(x.split('-')[-1]))
        last_checkpoint = checkpoints[-1]
        print(f"🔄 Tìm thấy checkpoint cũ: {last_checkpoint}")

base_model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
base_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q", "k", "v", "o", "wi_0", "wi_1", "wo"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=1e-3,
    optim="adafactor",
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    report_to="none",
    logging_steps=50,
    load_best_model_at_end=True,
    generation_max_length=128,
)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

if last_checkpoint:
    print(f"🚀 Tiếp tục train từ checkpoint LoRA: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("🚀 Bắt đầu train LoRA mới")
    trainer.train()

trainer.save_model(FINAL_SAVE_PATH)
tokenizer.save_pretrained(FINAL_SAVE_PATH)

# PHẦN 4: INFERENCE (DỰ ĐOÁN THEO BATCH SIÊU TỐC)
print("\n>>> [4/5] Chạy Inference (Batch Processing)...")
model.eval()
predictions, gold_lines = [], []
input_texts = []

for item in val_ds:
    input_texts.append(f"translate to SQL: {item['question']} | Schema: {get_crp_schema(item['db_id'])}")
    gold_lines.append(f"{item['query']}\t{item['db_id']}\n")

batch_size = 16
total_batches = len(input_texts)
print(f"Bắt đầu dịch {total_batches} câu hỏi sang SQL...")
start_time = time.time()

for i in range(0, total_batches, batch_size):
    batch_texts = input_texts[i : i + batch_size]

    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=1024)

    input_ids = inputs.input_ids.to(model.device)
    attention_mask = inputs.attention_mask.to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=128,
            num_beams=8,
            early_stopping=True
        )

    batch_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend([pred + "\n" for pred in batch_preds])

    print(f"\rĐã xử lý: {min(i + batch_size, total_batches)}/{total_batches}", end="")

print(f"\n✅ Xong Inference trong {time.time() - start_time:.2f} giây!")

with open('pred.txt', 'w') as f: f.writelines(predictions)
with open('gold.txt', 'w') as f: f.writelines(gold_lines)

# PHẦN 5: ĐÁNH GIÁ (EVALUATION)
print("\n>>> [5/5] Kết quả đánh giá:")
os.system('sed -i \'s/conn = sqlite3.connect(db)/conn = sqlite3.connect(db)\\n    conn.text_factory = lambda b: b.decode(errors="ignore")/\' evaluation.py')
os.system('python evaluation.py --gold gold.txt --pred pred.txt --db spider_data/database --table spider_data/tables.json --etype all')

✅ Đã whitelist Numpy cho PyTorch.
✅ Đã ép buộc torch.load(weights_only=False) thành công.
Mounted at /content/drive
>>> [1/5] Tải dữ liệu...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 6,684,672 || all params: 67,191,296 || trainable%: 9.9487
🚀 Tiếp tục train từ checkpoint LoRA: /content/drive/My Drive/T5_Small_Spider_CRP_LoRA/checkpoints/checkpoint-3285


Epoch,Training Loss,Validation Loss



>>> [4/5] Chạy Inference (Batch Processing)...
Bắt đầu dịch 1034 câu hỏi sang SQL...
Đã xử lý: 1034/1034
✅ Xong Inference trong 177.35 giây!

>>> [5/5] Kết quả đánh giá:


0

In [ ]:
print("\n>>> [5/5] Kết quả đánh giá:")
!sed -i 's/conn = sqlite3.connect(db)/conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors="ignore")/' evaluation.py
!python evaluation.py --gold gold.txt --pred pred.txt --db spider_data/database --table spider_data/tables.json --etype all


>>> [5/5] Kết quả đánh giá:
easy pred: SELECT count(*) FROM singer_in_concert
easy gold: SELECT count(*) FROM singer

easy pred: SELECT sum(T1.singer_id) FROM singer_in_concert AS T1 JOIN concert AS T2 ON T1.concert_id = T2.concert_id GROUP BY T1.concert_id
easy gold: SELECT count(*) FROM singer

eval_err_num:1
medium pred: SELECT T1.name, T1.country, T1.age FROM singer AS T1 JOIN singer_in_concert AS T2 ON T1.singer_id = T2.singer_id ORDER BY T2.age DESC
medium gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC

eval_err_num:2
medium pred: SELECT T1.name, T1.year FROM singer AS T1 JOIN singer_in_concert AS T2 ON T1.singer_id = T2.singer_id ORDER BY T2.age DESC LIMIT 1
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

medium pred: SELECT Song_release_year, Song_release_year FROM singer ORDER BY Age DESC LIMIT 1
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

eval_err_num:3
hard pred: SELECT T1.name 

In [2]:

# ĐÁNH GIÁ HIỆU NĂNG CHO MÔ HÌNH LORA (T5-Small)


import torch
import time
import numpy as np
import psutil
import os
import gc
from transformers import T5Tokenizer, T5ForConditionalGeneration
from peft import PeftModel

print("Đang dọn dẹp bộ nhớ GPU...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# 1. Đường dẫn mô hình
BASE_MODEL_NAME = "t5-small"
LORA_SAVE_PATH = "/content/drive/My Drive/T5_Small_Spider_CRP_LoRA"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Tải và Trộn mô hình (Merge)
print("Đang tải mô hình gốc và ghép nối trọng số LoRA...")
try:

    tokenizer = T5Tokenizer.from_pretrained(LORA_SAVE_PATH)
    base_model = T5ForConditionalGeneration.from_pretrained(BASE_MODEL_NAME)

    model = PeftModel.from_pretrained(base_model, LORA_SAVE_PATH)
    model = model.merge_and_unload()

    model.to(device)
    model.eval()
    print(" Tải và trộn mô hình LoRA thành công!")
except Exception as e:
    print(f" Lỗi khi tải mô hình: {e}")
    exit()

sample_question = "How many students are there?"
sample_schema = "CREATE TABLE student(id int, name text);"
input_text = f"translate to SQL: {sample_question} | Schema: {sample_schema}"

inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=1024).to(device)

# 4. Warmup GPU
print("Đang warmup GPU...")
for _ in range(10):
    with torch.no_grad():
        model.generate(**inputs, max_length=128)

# 5. Đo Latency (Độ trễ)
print("Đang đo lường độ trễ (100 lần chạy)...")
latencies = []
for _ in range(100):
    start = time.time()
    with torch.no_grad():
        model.generate(**inputs, max_length=128)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    latencies.append(time.time() - start)

avg_latency = np.mean(latencies)
std_latency = np.std(latencies)

# 6. Thông lượng (Throughput) Batch Size = 1
throughput = 1 / avg_latency

# 7. Đo VRAM và RAM
gpu_memory_allocated = None
if torch.cuda.is_available():
    gpu_memory_allocated = torch.cuda.max_memory_allocated() / 1024**2

process = psutil.Process(os.getpid())
ram_usage = process.memory_info().rss / 1024**2
total_params = sum(p.numel() for p in model.parameters())

# 8. In kết quả
print("\n" + "="*45)
print("     ĐÁNH GIÁ HIỆU NĂNG (T5-SMALL + LORA)      ")
print("="*45)
print(f"Độ trễ TB (Latency):            {avg_latency*1000:.2f} ms")
print(f"Độ lệch chuẩn (Std):            {std_latency*1000:.2f} ms")
print(f"Thông lượng (Throughput BS=1):  {throughput:.2f} samples/sec")
if gpu_memory_allocated:
    print(f"VRAM tối đa đã dùng (Peak):     {gpu_memory_allocated:.2f} MB")
print(f"RAM CPU đang sử dụng:           {ram_usage:.2f} MB")
print(f"Tổng số tham số mô hình:        {total_params:,}")
print("="*45)

Đang dọn dẹp bộ nhớ GPU...
Đang tải mô hình gốc và ghép nối trọng số LoRA...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

✅ Tải và trộn mô hình LoRA thành công!
Đang warmup GPU...
Đang đo lường độ trễ (100 lần chạy)...

     ĐÁNH GIÁ HIỆU NĂNG (T5-SMALL + LORA)      
Độ trễ TB (Latency):            174.75 ms
Độ lệch chuẩn (Std):            107.33 ms
Thông lượng (Throughput BS=1):  5.72 samples/sec
VRAM tối đa đã dùng (Peak):     289.94 MB
RAM CPU đang sử dụng:           1655.74 MB
Tổng số tham số mô hình:        60,506,624
